In [ ]:
# Googleドライブに対するマウントを設定
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ─────────────────────────────────────────────
# 標準ライブラリのインポート
# ─────────────────────────────────────────────
import os       # ファイルパスの操作に使う
import glob     # ワイルドカード(*) でファイルを一括検索するために使う
import smtplib  # メール送信（SMTP通信）に使う
from datetime import datetime, timezone, timedelta                   # 今日の日付・曜日の取得に使う
from email.mime.text import MIMEText            # メール本文を作るために使う
from email.mime.multipart import MIMEMultipart  # 件名・宛先などメール全体を作るために使う

# サードパーティライブラリのインポート
import openpyxl  # Excel ファイル (.xlsx) の読み書きに使う（pip install openpyxl でインストール）


# ─────────────────────────────────────────────
# 設定変数 (実行前に各自の情報に書き換えてください)
# ─────────────────────────────────────────────

# Mailtrap の SMTP 接続情報（メール送信に必要）
SMTP_HOST     = "sandbox.smtp.mailtrap.io"  # Mailtrap の SMTPサーバーアドレス
SMTP_PORT     = 587                          # SMTP のポート番号（587 = STARTTLS）
SMTP_USERNAME = "f63f8834abdfe0"             # Mailtrap のユーザー名
SMTP_PASSWORD = "443cb758f1fadd"             # Mailtrap のパスワード

# メールの送受信アドレス
SENDER_ADDRESS   = "from@example.com"  # 送信元メールアドレス
RECEIVER_ADDRESS = "to@example.com"   # 送信先メールアドレス（農家の担当者など）

# ファイルパス
ORDER_PATTERN   = "/content/drive/MyDrive/PGスクール/samples/order_new/order_*.xlsx"
INVENTORY_FILE  = "/content/drive/MyDrive/PGスクール/samples/inventory.xlsx"
PICKUP_FILE     = "/content/drive/MyDrive/PGスクール/samples/pickup.xlsx"

# タイムゾーン
JST = timezone(timedelta(hours=9))


# ─────────────────────────────────────────────
# Step 1: 各店からの注文をまとめて合計を計算する
# ─────────────────────────────────────────────
def aggregate_orders(order_pattern: str) -> dict[str, int]:
    """
    order_*.xlsx を全て読み込み、野菜ごとの合計注文数（整数）を返す。

    引数:
        order_pattern: 注文ファイルのグロブパターン（例: "/path/to/order_*.xlsx"）
    戻り値:
        {野菜名: 合計注文数} の辞書
        例: {"トマト": 13, "レタス": 23}

    ファイル形式:
        1行目 = 野菜名（ヘッダー）
        2行目 = 注文数
    """
    # 合計注文数を格納する辞書（最初は空）
    total_orders: dict[str, int] = {}

    # glob.glob() でパターンに一致するファイルを全て取得する
    # 例: ["order_A_20230524.xlsx", "order_B_20230524.xlsx"]
    order_files = glob.glob(order_pattern)

    # 注文ファイルが1件も見つからない場合はエラーを出して処理を止める
    if not order_files:
        raise FileNotFoundError(f"注文ファイルが見つかりません: {order_pattern}")

    # 各注文ファイルを1件ずつ読み込む
    for filepath in order_files:
        wb = openpyxl.load_workbook(filepath)  # Excelファイルを開く
        ws = wb.active                          # 先頭のシートを取得する

        # 1行目のセルを左から順に読み込み、値が入っているセルだけリストにする
        # 例: ["トマト", "キャベツ", "レタス", ...]
        headers = [cell.value for cell in ws[1] if cell.value is not None]

        # 2行目のセルを左から順に読み込み、値が入っているセルだけリストにする
        # 例: [13, 6, 11, ...]
        values = [cell.value for cell in ws[2] if cell.value is not None]

        # 野菜名と注文数をペアにして合計に加算する
        # zip(headers, values) で [("トマト", 13), ("キャベツ", 6), ...] のように組み合わせる
        for name, qty in zip(headers, values):
            # すでに登録済みの野菜は加算、未登録の野菜は 0 から加算する
            # int() で整数に変換（Excelから読み込んだ値は float になる場合がある）
            total_orders[name] = total_orders.get(name, 0) + int(qty)

    # 集計結果をコンソールに表示する
    print("【Step 1】注文集計完了")
    for veg, qty in total_orders.items():
        print(f"  {veg}: {qty}")

    return total_orders


# ─────────────────────────────────────────────
# Step 2: inventory.xlsx から最新在庫を取得する
# ─────────────────────────────────────────────
def get_latest_inventory(inventory_file: str) -> tuple[list[str], dict[str, int]]:
    """
    inventory.xlsx の最終行（= 最新日付のデータ）を読み取り、在庫数を返す。

    引数:
        inventory_file: inventory.xlsx のファイルパス
    戻り値:
        (列ヘッダーのリスト, {野菜名: 在庫数} の辞書) のタプル
        例: (["日付", "曜日", "トマト", ...], {"トマト": 91, "キャベツ": 73, ...})

    inventory.xlsx の列構成:
        A列=日付, B列=曜日, C列以降=野菜ごとの在庫数
    """
    wb = openpyxl.load_workbook(inventory_file)  # Excelファイルを開く
    ws = wb.active                                # 先頭のシートを取得する

    # 1行目（ヘッダー行）を全列分読み込む
    # None も含めて取得する（後で列の順番を保持するために必要）
    # 例: ["日付", "曜日", "トマト", "キャベツ", ...]
    headers = [cell.value for cell in ws[1]]

    # 2行目以降を順に確認して、日付が入っている最後の行を取得する
    # これが「最新の在庫データ」になる
    last_row = None
    for row in ws.iter_rows(min_row=2):  # 2行目から最終行まで1行ずつ確認
        if row[0].value is not None:     # A列（日付列）に値があれば有効な行とみなす
            last_row = row               # 有効な行を上書きし続けることで最終行が残る

    # データが1行も見つからない場合はエラーを出して処理を止める
    if last_row is None:
        raise ValueError("inventory.xlsx にデータがありません。")

    # C列以降（インデックス2以降）が野菜の在庫数なので、野菜名と在庫数を辞書にまとめる
    vegetable_headers = headers[2:]  # ["トマト", "キャベツ", "レタス", ...]
    inventory: dict[str, int] = {}
    for header, cell in zip(vegetable_headers, last_row[2:]):
        if header is not None:
            # int() で整数に変換する。値が空の場合は 0 とする
            inventory[header] = int(cell.value) if cell.value is not None else 0

    # 確認用にコンソールへ表示する
    print("\n【Step 2】最新在庫確認完了")
    for veg, qty in inventory.items():
        print(f"  {veg}: {qty}")

    # headers はStep6でExcelに書き込む際の列順序として必要なので一緒に返す
    return headers, inventory


# ─────────────────────────────────────────────
# Step 3: 発注が必要なアイテムを特定する
# ─────────────────────────────────────────────
def identify_orders_needed(
    inventory: dict[str, int],
    total_orders: dict[str, int],
    pickup_file: str,
) -> dict[str, int]:
    """
    在庫 − 注文数 を計算し、しきい値を下回った野菜の発注量を返す。

    引数:
        inventory   : {野菜名: 現在庫数} の辞書（Step2の結果）
        total_orders: {野菜名: 合計注文数} の辞書（Step1の結果）
        pickup_file : pickup.xlsx のファイルパス
    戻り値:
        {野菜名: 発注量} の辞書（発注不要な野菜は含まれない）
        例: {"ニンジン": 80}

    pickup.xlsx の行構成:
        1行目 = ヘッダー（野菜名）
        2行目 = しきい値（在庫がこの数を下回ったら発注する）
        3行目 = 発注量（発注する個数）
    """
    wb = openpyxl.load_workbook(pickup_file)  # pickup.xlsx を開く
    ws = wb.active

    # 1行目から野菜名の一覧を取得する
    # 例: ["トマト", "キャベツ", ...]
    col_headers = [cell.value for cell in ws[1] if cell.value is not None]

    # 2行目からしきい値を取得する。list(ws[2])[1:] でB列以降を取得（A列はラベルなので除外）
    # 結果例: {"トマト": 50, "キャベツ": 40, ...}
    thresholds = {col: int(cell.value) for col, cell in zip(col_headers, list(ws[2])[1:])}

    # 3行目から発注量を取得する（同じくB列以降）
    # 結果例: {"トマト": 100, "キャベツ": 80, ...}
    additions = {col: int(cell.value) for col, cell in zip(col_headers, list(ws[3])[1:])}

    # 注文を引いた後の在庫（= 注文後在庫）を計算する
    # .get(veg, 0) は「辞書に該当キーが無ければ 0 を返す」という意味
    after_order: dict[str, int] = {}
    for veg, current_qty in inventory.items():
        ordered_qty = total_orders.get(veg, 0)       # その野菜の合計注文数（なければ0）
        after_order[veg] = current_qty - ordered_qty  # 注文後の残在庫

    # 注文後在庫 vs しきい値を比較して、発注が必要な野菜を特定する
    print("\n【Step 3】注文後在庫 vs しきい値")
    orders_needed: dict[str, int] = {}
    for veg, remaining in after_order.items():
        threshold = thresholds.get(veg, 0)  # しきい値（なければ0）
        addition  = additions.get(veg, 0)   # 発注量（なければ0）
        status = "発注不要"
        if remaining < threshold:
            # しきい値を下回った場合は発注対象に追加する
            orders_needed[veg] = addition
            status = f"発注必要 (発注量: {addition})"
        print(f"  {veg}: 残量={remaining}, しきい値={threshold} {status}")

    return orders_needed


# ─────────────────────────────────────────────
# Step 4: 発注メール文を作成する
# ─────────────────────────────────────────────
def create_email_body(orders_needed: dict[str, int]) -> str:
    """
    発注が必要な野菜のリストからメール本文を作成して文字列で返す。

    引数:
        orders_needed: {野菜名: 発注量} の辞書（Step3の結果）
    戻り値:
        メール本文の文字列
    """
    # 今日の日付を「2026年03月15日」形式の文字列にする
    today_str = datetime.now(JST).strftime("%Y年%m月%d日")

    # 発注品目を箇条書き形式のテキストに変換する
    lines = []
    for veg, qty in orders_needed.items():
        lines.append(f"  ・{veg}: {qty} 個")

    # 発注品目が1件もない場合の表示（通常は呼ばれないが念のため）
    order_list = "\n".join(lines) if lines else "  （発注なし）"

    # f文字列（fstring）を使って変数をメール本文に埋め込む
    # 三重クォート（"""）で複数行のテキストをそのまま書ける
    body = f"""
農家 ご担当者様

お世話になっております。
野菜卸売業 担当者です。

本日（{today_str}）の在庫確認の結果、
下記の商品について補充のお願いをしたく、ご連絡差し上げます。

【発注品目】
{order_list}

お手数ですが、早急にご対応いただけますと幸いです。
何卒よろしくお願い申し上げます。

野菜卸売業
担当者
"""
    # strip() で先頭・末尾の余分な改行を取り除く
    return body.strip()


# ─────────────────────────────────────────────
# Step 5: メールを送信する (Mailtrap SMTP)
# ─────────────────────────────────────────────
def send_email(body: str) -> None:
    """
    Mailtrap SMTP 経由でメールを送信する。
    送信先・認証情報はスクリプト上部の設定変数を使用する。

    引数:
        body: 送信するメール本文（Step4の結果）
    """
    # MIMEMultipart でメールの「封筒」を作成する（件名・宛先・本文をまとめる入れ物）
    msg = MIMEMultipart()
    msg["From"]    = SENDER_ADDRESS    # 送信元アドレス
    msg["To"]      = RECEIVER_ADDRESS  # 送信先アドレス
    # 件名に今日の日付を含める
    msg["Subject"] = f"【発注依頼】在庫補充のお願い ({datetime.now(JST).strftime('%Y-%m-%d')})"

    # 本文を UTF-8 のテキストとして添付する
    msg.attach(MIMEText(body, "plain", "utf-8"))

    try:
        # with ブロックを使うことで、処理が終わったら自動的に接続を閉じる
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()                               # サーバーへ自己紹介（SMTP の手順）
            server.starttls()                           # 通信を暗号化（TLS）に切り替える
            server.login(SMTP_USERNAME, SMTP_PASSWORD)  # 認証情報でログインする
            server.sendmail(SENDER_ADDRESS, RECEIVER_ADDRESS, msg.as_string())  # メール送信
        print("\n【Step 5】メール送信完了")
    except Exception as e:
        # 送信に失敗した場合はエラー内容を表示して例外を再送出する
        print(f"\n【Step 5】メール送信失敗: {e}")
        raise


# ─────────────────────────────────────────────
# Step 6: inventory.xlsx に発注後の在庫を追記する
# ─────────────────────────────────────────────
def update_inventory(
    inventory_file: str,
    headers: list[str],
    inventory: dict[str, int],
    total_orders: dict[str, int],
    orders_needed: dict[str, int],
) -> None:
    """
    発注後の在庫数（在庫 − 注文数 ＋ 発注量）を本日日付で
    inventory.xlsx の末尾に追記し、上書き保存する。

    引数:
        inventory_file: inventory.xlsx のファイルパス
        headers       : 列ヘッダーのリスト（Step2の結果）
        inventory     : {野菜名: 現在庫数} の辞書（Step2の結果）
        total_orders  : {野菜名: 合計注文数} の辞書（Step1の結果）
        orders_needed : {野菜名: 発注量} の辞書（Step3の結果）
    """
    # 今日の日付と曜日を取得する
    today = datetime.now(JST)
    # today.weekday() は月曜=0, 火曜=1, ... 日曜=6 を返すので、リストで対応表を作る
    day_jp = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][today.weekday()]

    # 発注後の在庫を計算する
    # 計算式: 在庫 − 注文数 ＋ 発注量
    #   ・注文数 : 今日の合計注文数（orders_needed に含まれない野菜は 0）
    #   ・発注量 : しきい値を下回った野菜だけ加算（下回っていない野菜は 0）
    new_inventory: dict[str, int] = {}
    for veg, current_qty in inventory.items():
        ordered_qty = total_orders.get(veg, 0)   # 今日の注文数（なければ 0）
        pickup_qty  = orders_needed.get(veg, 0)  # 発注量（発注不要な野菜は 0）
        new_inventory[veg] = current_qty - ordered_qty + pickup_qty

    # inventory.xlsx を開き、末尾に新しい行を追記する
    wb = openpyxl.load_workbook(inventory_file)
    ws = wb.active

    # headers の順番（日付・曜日・トマト・...）に合わせて新しい行のデータを作成する
    new_row = []
    for col in headers:
        if col == "日付":
            # timezone情報を取り除いてから追加
            new_row.append(today.replace(tzinfo=None))
        elif col == "曜日":
            new_row.append(day_jp)             # 曜日の文字列（"Mon" など）を追加
        elif col in new_inventory:
            new_row.append(new_inventory[col]) # 発注後の在庫数（整数）を追加
        else:
            new_row.append(None)               # 対応する野菜がない列は空欄にする

    ws.append(new_row)      # シートの末尾に新しい行を追加する
    wb.save(inventory_file) # ファイルを上書き保存する

    # 更新後の在庫をコンソールに表示する（発注補充があった場合はその旨も表示）
    print("\n【Step 6】在庫更新完了 → inventory.xlsx に追記しました")
    for veg, qty in new_inventory.items():
        pickup_qty  = orders_needed.get(veg, 0)
        # 発注補充があった野菜は補充量も表示する
        pickup_info = f"  (発注補充 +{pickup_qty})" if pickup_qty > 0 else ""
        print(f"  {veg}: {qty}{pickup_info}")


# ─────────────────────────────────────────────
# メイン処理：各ステップを順番に呼び出す
# ─────────────────────────────────────────────
def main():
    print("=" * 50)
    print("在庫管理・発注自動化スクリプト 開始")
    print("=" * 50)

    # Step 1: order_*.xlsx を全件読み込んで野菜ごとの合計注文数を計算する
    total_orders = aggregate_orders(ORDER_PATTERN)

    # Step 2: inventory.xlsx の最終行から現在の在庫数を取得する
    headers, inventory = get_latest_inventory(INVENTORY_FILE)

    # Step 3: しきい値と比較して発注が必要な野菜と発注量を特定する
    orders_needed = identify_orders_needed(inventory, total_orders, PICKUP_FILE)

    # 発注が必要な野菜がある場合だけ Step 4・5 を実行する
    if not orders_needed:
        print("\n発注が必要な野菜はありません。メールは送信しません。")
    else:
        # Step 4: 発注品目をメール本文に整形する
        email_body = create_email_body(orders_needed)
        print(f"\n【Step 4】メール本文作成完了:\n{'-'*40}\n{email_body}\n{'-'*40}")

        # Step 5: 作成したメールを農家へ送信する
        send_email(email_body)

    # Step 6: 発注後の在庫数（在庫 − 注文数 ＋ 発注量）を inventory.xlsx に追記する
    # ※ 発注の有無にかかわらず毎回実行する
    update_inventory(INVENTORY_FILE, headers, inventory, total_orders, orders_needed)

    print("\n" + "=" * 50)
    print("全処理完了")
    print("=" * 50)


# ─────────────────────────────────────────────
# エントリーポイント
# ─────────────────────────────────────────────
# このファイルを直接実行したときだけ main() を呼び出す
# （他のファイルから import されたときは呼び出されない）
if __name__ == "__main__":
    main()

在庫管理・発注自動化スクリプト 開始
【Step 1】注文集計完了
  トマト: 31
  キャベツ: 21
  レタス: 42
  白菜: 25
  ほうれん草: 23
  大根: 15
  ニンジン: 32

【Step 2】最新在庫確認完了
  トマト: 60
  キャベツ: 52
  レタス: 61
  白菜: 59
  ほうれん草: 52
  大根: 33
  ニンジン: 98

【Step 3】注文後在庫 vs しきい値
  トマト: 残量=29, しきい値=50 発注必要 (発注量: 100)
  キャベツ: 残量=31, しきい値=40 発注必要 (発注量: 80)
  レタス: 残量=19, しきい値=50 発注必要 (発注量: 100)
  白菜: 残量=34, しきい値=30 発注不要
  ほうれん草: 残量=29, しきい値=40 発注必要 (発注量: 80)
  大根: 残量=18, しきい値=30 発注必要 (発注量: 60)
  ニンジン: 残量=66, しきい値=40 発注不要

【Step 4】メール本文作成完了:
----------------------------------------
農家 ご担当者様

お世話になっております。
野菜卸売業 担当者です。

本日（2026年03月17日）の在庫確認の結果、
下記の商品について補充のお願いをしたく、ご連絡差し上げます。

【発注品目】
  ・トマト: 100 個
  ・キャベツ: 80 個
  ・レタス: 100 個
  ・ほうれん草: 80 個
  ・大根: 60 個

お手数ですが、早急にご対応いただけますと幸いです。
何卒よろしくお願い申し上げます。

野菜卸売業
担当者
----------------------------------------

【Step 5】メール送信完了

【Step 6】在庫更新完了 → inventory.xlsx に追記しました
  トマト: 129  (発注補充 +100)
  キャベツ: 111  (発注補充 +80)
  レタス: 119  (発注補充 +100)
  白菜: 34
  ほうれん草: 109  (発注補充 +80)
  大根: 78  (発注補充 +60)
  ニンジン: 66

全処理完了
